Note: 

1. This is a variable length sliding window example 
2. Is this "maximum length subarray" problem ? 


A bank is piloting an AI pull-request reviewer. For each review run, the platform records the tools the agent called, in execution order. The platform team wants to find the longest contiguous stretch of calls that uses at most k distinct tools. An engineer used an LLM to draft the function below and asks you to review it before merging.

The contract
- Return (start, end) using zero-based indices and an exclusive end. If multiple stretches have the same maximum length, return the earliest.
- Return (0, 0) for empty input or k == 0.
- 0 <= len(calls) <= 200_000, k >= 0, and tool names are nonempty strings. Preserve the input. The caller expects average O(n) time.

Acceptance examples

| `calls` | `k` | Expected |
|---|---:|---:|
| `["diff", "policy", "diff", "tests", "tests", "tests"]` | 2 | `(2, 6)` |
| `["diff", "policy", "tests"]` | 2 | `(0, 2)` |
| `["diff"]` | 0 | `(0, 0)` |

Your task: Would you approve this implementation? Give your review as you would live: demonstrate a contract violation with a concrete input, submit the corrected function with focused tests, and explain its time and auxiliary-space complexity. If you use AI assistance, include one suggestion you independently verified.

In [ ]:
def longest_tool_span(calls: list[str], k: int) -> tuple[int, int]:
    counts = {}
    left = 0
    best = (0, 0)

    if not calls or k == 0:
        return (0, 0)

    for right, tool in enumerate(calls):
        counts[tool] = counts.get(tool, 0) + 1

        if len(counts) > k:
            outgoing = calls[left]
            counts[outgoing] -= 1
            if counts[outgoing] == 0:
                del counts[outgoing]
            left += 1
        
        if right + 1 - left >= best[1] - best[0]:
            best = (left, right + 1)

    return best

# Usage 
result = longest_tool_span([], 2)
print (f"tc #1: {result}")
print () 

result = longest_tool_span(['diff'], 0)
print (f"tc #2: {result}")
print () 

calls = ["diff", "policy", "tests"]
k = 2

result = longest_tool_span(calls, k)
print (result)

tc #1: (0, 0)

0 diff 1 {'diff': 1}
1 0 (1, 1)

tc #2: (1, 1)

0 diff 1 {'diff': 1}
0 0 (0, 1)

1 policy 2 {'diff': 1, 'policy': 1}
0 1 (0, 2)

2 tests 3 {'diff': 1, 'policy': 1, 'tests': 1}
1 2 (1, 3)

(1, 3)


1. Contract violations 

Violation A. Using if instead of while when shrinking the window

In a variable-size sliding window, moving left forward by just one step does not guarantee that the number of distinct tools drops back to <= k. If the element at calls[left] had a frequency greater than 1 in the current window, len(counts) remains k + 1, violating the at most k distinct tools invariant.

• Concrete failing input: calls = ["a", "a", "b", "c"], k = 2


Violation B: Off-by-one tie-breaking (>= instead of >)

The contract requires returning the earliest stretch when multiple stretches share the same maximum length. Using >= overwrites best whenever a later window ties the current maximum length.

• Concrete failing input: calls = ["diff", "policy", "tests"], k = 2 (from the acceptance examples)
• Execution trace:
	• right = 1: Window ["diff", "policy"] has length 2. best becomes (0, 2).
	• right = 2: Window shrinks to ["policy", "tests"], which also has length 2. Because 2 >= 2 is true, best is overwritten with (1, 3).
• Actual output: (1, 3)
• Expected output: (0, 2)


Corrected Implmentation

In [ ]:
def longest_tool_span(calls: list[str], k: int) -> tuple[int, int]:
    if not calls or k <= 0:
        return (0, 0)

    counts: dict[str, int] = {}
    left = 0
    best_start = 0
    best_len = 0

    for right, tool in enumerate(calls):
        counts[tool] = counts.get(tool, 0) + 1

        while len(counts) > k:
            outgoing = calls[left]
            counts[outgoing] -= 1
            if counts[outgoing] == 0:
                del counts[outgoing]
            left += 1

        window_len = right - left + 1
        if window_len > best_len:
            best_len = window_len
            best_start = left

    return (best_start, best_start + best_len)


0

In [ ]:
def test_longest_tool_span():
    # 1. Acceptance examples
    assert longest_tool_span(["diff", "policy", "diff", "tests", "tests", "tests"], 2) == (2, 6)
    assert longest_tool_span(["diff", "policy", "tests"], 2) == (0, 2)
    assert longest_tool_span(["diff"], 0) == (0, 0)

    # 2. Regression: Repeated tool at left boundary requiring multi-step shrink
    assert longest_tool_span(["a", "a", "b", "c"], 2) == (0, 3)
    assert longest_tool_span(["a", "a", "a", "b", "c", "c", "c"], 2) == (3, 7)

    # 3. Regression: Tie-breaking favors the earliest window
    assert longest_tool_span(["a", "b", "c", "d"], 1) == (0, 1)
    assert longest_tool_span(["a", "b", "b", "c", "c"], 2) == (0, 3)

    # 4. Edge cases: Empty input, k >= distinct tools, single element
    assert longest_tool_span([], 3) == (0, 0)
    assert longest_tool_span([], 0) == (0, 0)
    assert longest_tool_span(["diff", "policy"], 5) == (0, 2)
    assert longest_tool_span(["diff"], 1) == (0, 1)

    # 5. Input preservation check
    original = ["diff", "policy", "tests"]
    snapshot = list(original)
    longest_tool_span(original, 2)
    assert original == snapshot



. Complexity Analysis

• Time Complexity: O(n) average / O(n) amortized, where n = len(calls).
	• The right pointer advances from 0 to n - 1 exactly once (n iterations).
	• The left pointer only ever moves forward and can advance at most n times across the entire execution of the function.
	• Each element is added to counts once and removed at most once; dictionary insertions, lookups, and deletions take O(1) average time. For n ≤ 200,000, this executes well within a few milliseconds.
• Auxiliary Space Complexity: \(O(\min(n, k))\).
	• The input list calls is read-only and never copied or mutated.
	• The counts hash map stores at most k + 1 keys at any instant before the while loop evicts one, and never more than n keys if k ≥ n. Tracking best_start and best_len uses O(1) scalar memory.



4. Independently Verified AI Review Note

• Verification of the while vs. if subtle trap: In classic "maximum length subarray" problems where you only care about the scalar length (not the indices or validity at every step), an if-based non-shrinking sliding window trick is sometimes used because the window never shrinks below the max size found so far. However, I independently verified that that trick breaks this contract: because we must return exact (start, end) indices and preserve earliest-tie semantics, allowing left to lag leaves left pointing inside an invalid window (as shown with ["a", "a", "b", "c"], k = 2, where best gets corrupted to (1, 4)). Using the while loop is strictly required here to keep (left, right + 1) valid on every iteration.
